# Welcome!

This is likely the first file you are looking at (or at least it should be). Have a lovely stay parsing through!

I needed a good way of downloading the data as: There are a lot of different files, versions, and organization schemes for all of the datasets, and some of the files are LARGE, so without a proper download management utility, it is likely to fail and be incomplete. Additionally, I moved computers, and needed to reformat the only drive I had out of my old one into my new, and with current storage pricing, I could not get a new drive. Trust me, writing this code was easier than managing all this spaghetti myself

# Music data downloader: MetaBrainz + Kaggle + Hugging Face

Downloads training data for my music recommendation system, checking for files that already exist locally (skipping or resuming them) and only fetching what's missing or changed. Directories are created automatically. MetaBrainz dumps that live inside a timestamped folder on the server (e.g. `YYYYMMDD-HHMMSS/`) are downloaded into a matching local staging folder and then moved into a stable, un-timestamped folder once complete, so downstream tools always have a fixed path to point at.

This notebook downloads everything into the data folder: the folder in the environment variable `THESIS_DATA_ROOT`, or else `data/` next to the notebook. The pipeline reads the same variable (`pipeline/config.py`). `BASE_DIR` below is not hardcoded, so it isn't tied to one absolute path across machines.

## Sources

| Source | What it is | Approx. size |
|---|---|---|
| MusicBrainz full export (PostgreSQL dumps) | Item metadata: artists, releases, recordings | ~23 GB |
| MusicBrainz JSON dumps | Same data, JSON format | ~24 GB |
| ListenBrainz full export | User listen events (interaction data) | ~191 GB |
| ListenBrainz incremental dumps | Daily deltas on top of the full export | tens of GB, growing daily |
| CritiqueBrainz dumps (Postgres + JSON) | User-written text reviews | tens of MB |
| MusicBrainz canonical data dump | Deduplicated artist/recording ID mapping | ~2 GB |
| MLHD+ | Anonymized listening histories | ~272 GB |
| Kaggle: `lordpatil/spotify-metadata-by-annas-archive` | The raw Spotify catalogue. artists, albums, tracks, playlists, and the links between them | ~176 GB |
| Hugging Face: `ozefe/spotify_audio_features` | Spotify's own per-track API metadata: audio features (tempo, key, danceability, energy...) and popularity, one row per track | ~12.1 GB |

**Dry run** cell runs first. It lists exact file counts and sizes for everything, with zero data transferred. Then the download cell runs only if `CONFIRM_DOWNLOAD = True`.

Downloads are resumable. If the notebook is interrupted, just re-run it.
Already complete files are skipped and partial downloads resume from where they left off.

In [ ]:
# Dependencies, at the versions of requirements.txt
import sys

!{sys.executable} -m pip install -q -r requirements.txt

In [ ]:
import os
import shutil
import hashlib
from pathlib import Path
from urllib.parse import urljoin

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
from tqdm import tqdm
from huggingface_hub import HfApi, snapshot_download

NOTEBOOK_DIR = Path(os.path.abspath(""))
BASE_DIR = Path(os.environ.get("THESIS_DATA_ROOT", NOTEBOOK_DIR / "data"))
BASE_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 8 * 1024 * 1024  # 8 MB

SESSION = requests.Session()
_retries = Retry(
    total=5, backoff_factor=1.0, status_forcelist=[429, 500, 502, 503, 504]
)
SESSION.mount("https://", HTTPAdapter(max_retries=_retries))
SESSION.mount("http://", HTTPAdapter(max_retries=_retries))
SESSION.headers.update(
    {
        # MetaBrainz asks clients to identify themselves with contact info:
        # put an email address or a GitHub URL in place of CONTACT before a long run.
        "User-Agent": "music-data-downloader/1.0 (CONTACT)"
    }
)

print(f"Downloading into: {BASE_DIR}")

## Source configuration. 
Every entry has a "kind" that decides which download mechanism handles it:

`metabrainz`: data.metabrainz.org mirror (plain directory listings), using the
mode/flatten system described below

`kaggle`: Kaggle API, requires a Kaggle API token (see the credentials check
further down)

`huggingface`: Hugging Face Hub, public dataset, no token required

MetaBrainz-specific "mode" values:
`latest_pointer`: directory has a LATEST file containing the name of the current
dated subdirectory (musicbrainz full export, json dumps)
`max_subdir`: no LATEST file; pick the lexicographically-last dated subdirectory
(dated/versioned names sort correctly as strings here)
`all_subdirs`: download every dated subdirectory found (listenbrainz incremental)
`flat`: files sit directly in the directory, no versioned subfolder (MLHD+,
critiquebrainz json)

MetaBrainz-specific `flatten_after_download`:
When the target resolves to a versioned subdirectory, download files into a matching
staging folder, then move them up into the source's plain local_dir once the whole
version has downloaded successfully, and record the version in a ".version" marker
file. On the next run, if the marker already matches the current remote version, the
whole target is skipped without even listing files.

In [ ]:
SOURCES = {
    # MetaBrainz mirror
    "musicbrainz_fullexport": {
        "kind": "metabrainz",
        "url": "https://data.metabrainz.org/pub/musicbrainz/data/fullexport/",
        "local_dir": "musicbrainz/fullexport",
        "mode": "latest_pointer",
        "flatten_after_download": True,
        "enabled": True,
    },
    "musicbrainz_json_dumps": {
        "kind": "metabrainz",
        "url": "https://data.metabrainz.org/pub/musicbrainz/data/json-dumps/",
        "local_dir": "musicbrainz/json-dumps",
        "mode": "latest_pointer",
        "flatten_after_download": True,
        "enabled": True,
    },
    "listenbrainz_fullexport": {
        "kind": "metabrainz",
        "url": "https://data.metabrainz.org/pub/musicbrainz/listenbrainz/fullexport/",
        "local_dir": "listenbrainz/fullexport",
        "mode": "max_subdir",
        "flatten_after_download": True,
        "enabled": False,
    },
    "listenbrainz_incremental": {
        "kind": "metabrainz",
        "url": "https://data.metabrainz.org/pub/musicbrainz/listenbrainz/incremental/",
        "local_dir": "listenbrainz/incremental",
        "mode": "all_subdirs",
        "flatten_after_download": False,
        "enabled": False,  # Not that useful imo.
    },
    "critiquebrainz_dump": {
        "kind": "metabrainz",
        "url": "https://data.metabrainz.org/pub/musicbrainz/critiquebrainz/dump/",
        "local_dir": "critiquebrainz/dump",
        "mode": "max_subdir",
        "flatten_after_download": True,
        "enabled": False,
    },
    "critiquebrainz_json": {
        "kind": "metabrainz",
        "url": "https://data.metabrainz.org/pub/musicbrainz/critiquebrainz/json/",
        "local_dir": "critiquebrainz/json",
        "mode": "flat",
        "flatten_after_download": False,
        "enabled": False,
    },
    "canonical_data": {
        "kind": "metabrainz",
        "url": "https://data.metabrainz.org/pub/musicbrainz/canonical_data/",
        "local_dir": "canonical_data",
        "mode": "max_subdir",
        "flatten_after_download": True,
        "enabled": True,
    },
    "mlhd_plus": {
        "kind": "metabrainz",
        "url": "https://data.metabrainz.org/pub/musicbrainz/listenbrainz/mlhd/",
        "local_dir": "mlhd",
        "mode": "flat",
        "flatten_after_download": False,
        "enabled": False,
    },
    # Kaggle: the raw Spotify catalogue (artists, albums, tracks, playlists and their links)
    "kaggle_spotify_catalogue": {
        "kind": "kaggle",
        "kaggle_dataset": "lordpatil/spotify-metadata-by-annas-archive",
        "local_dir": "kaggle/spotify-catalogue",
        "enabled": False,
    },
    # Hugging Face: per-track Spotify API metadata (audio features and popularity)
    "huggingface_audio_features": {
        "kind": "huggingface",
        "hf_repo_id": "ozefe/spotify_audio_features",
        "local_dir": "huggingface/spotify-audio-features",
        "enabled": False,
    },
}

## Helper functions

MetaBrainz directory listings are plain nginx/Apache autoindex pages, parsed with
BeautifulSoup by reading the `<a href="...">` links. Kaggle and Hugging Face each have
their own client libraries, used directly.

In [ ]:
def human_size(n):
    if n is None:
        return "?"
    n = float(n)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024:
            return f"{n:.1f}{unit}"
        n /= 1024
    return f"{n:.1f}PB"


def list_directory(url):
    # Returns [{"name", "url", "is_dir"}, ...] for a directory-listing page, skipping the
    # parent-directory link.
    r = SESSION.get(url, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    entries = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("?") or href in ("../", "..") or href.startswith("/"):
            continue
        entries.append(
            {
                "name": href.rstrip("/"),
                "url": urljoin(url, href),
                "is_dir": href.endswith("/"),
            }
        )
    return entries


def remote_size(url):
    r = SESSION.head(url, allow_redirects=True, timeout=30)
    r.raise_for_status()
    cl = r.headers.get("Content-Length")
    return int(cl) if cl is not None else None


def resolve_targets(cfg):
    # Returns a list of {"url": directory_url, "version": name_or_None} to download files from.
    base_url = cfg["url"]
    mode = cfg["mode"]

    if mode == "flat":
        return [{"url": base_url, "version": None}]

    if mode == "latest_pointer":
        r = SESSION.get(urljoin(base_url, "LATEST"), timeout=30)
        r.raise_for_status()
        version = r.text.strip()
        return [{"url": urljoin(base_url, version + "/"), "version": version}]

    entries = list_directory(base_url)
    subdirs = sorted(e["name"] for e in entries if e["is_dir"])
    if not subdirs:
        return [{"url": base_url, "version": None}]

    if mode == "max_subdir":
        latest = subdirs[-1]
        return [{"url": urljoin(base_url, latest + "/"), "version": latest}]

    if mode == "all_subdirs":
        return [{"url": urljoin(base_url, s + "/"), "version": s} for s in subdirs]

    raise ValueError(f"Unknown mode: {mode}")

In [ ]:
def download_file(url, dest_path, expected_size=None, desc=None):
    # Resumable download: partial data is kept in a ".part" file and continued with an
    # HTTP Range request; only renamed to the final name once complete.
    dest_path = Path(dest_path)
    tmp_path = dest_path.with_name(dest_path.name + ".part")
    headers, mode, initial = {}, "wb", 0
    if tmp_path.exists():
        initial = tmp_path.stat().st_size
        headers["Range"] = f"bytes={initial}-"
        mode = "ab"

    with SESSION.get(url, headers=headers, stream=True, timeout=60) as r:
        if r.status_code == 416:
            tmp_path.rename(dest_path)
            return
        r.raise_for_status()
        total = expected_size
        if total is None:
            cl = r.headers.get("Content-Length")
            total = (int(cl) + initial) if cl else None
        with open(tmp_path, mode) as f, tqdm(
            total=total,
            initial=initial,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=desc or dest_path.name,
            leave=False,
        ) as bar:
            for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

    tmp_path.rename(dest_path)


def _prune_stale_files(final_dir, keep_names):
    # When a source's manifest changes between versions (MetaBrainz has occasionally
    # renamed or dropped files across dumps), a flattened directory can otherwise end up
    # with leftover files from the old version sitting alongside the new ones, with
    # nothing to tell them apart, since they all live at the same un-versioned path. Only
    # ever removes plain files it doesn't recognize; never touches subdirectories or the
    # .version marker itself.
    for p in final_dir.iterdir():
        if p.is_file() and p.name != ".version" and p.name not in keep_names:
            p.unlink()


def sync_target(cfg, target, dry_run=False):
    local_root = BASE_DIR / cfg["local_dir"]
    version = target["version"]
    flatten = cfg.get("flatten_after_download", False) and version is not None
    staging_dir = (local_root / version) if version else local_root
    final_dir = local_root if flatten else staging_dir
    marker_path = final_dir / ".version"

    # Fast path: this exact version was already fully downloaded and flattened before.
    if flatten and marker_path.exists() and marker_path.read_text().strip() == version:
        return {
            "version": version,
            "files": 0,
            "bytes": 0,
            "to_download": [],
            "up_to_date": True,
        }

    files = [e for e in list_directory(target["url"]) if not e["is_dir"]]
    remote_names = {f["name"] for f in files}

    to_download = []
    total_bytes = 0
    for f in files:
        final_path = final_dir / f["name"]
        size = remote_size(f["url"])
        if (
            final_path.exists()
            and size is not None
            and final_path.stat().st_size == size
        ):
            continue  # already have this exact file
        to_download.append({**f, "size": size})
        total_bytes += size or 0

    result = {
        "version": version,
        "files": len(to_download),
        "bytes": total_bytes,
        "to_download": to_download,
        "up_to_date": False,
    }

    if dry_run:
        return result

    if not to_download:
        if flatten:
            final_dir.mkdir(parents=True, exist_ok=True)
            _prune_stale_files(final_dir, remote_names)
            marker_path.write_text(version)
        return result

    staging_dir.mkdir(parents=True, exist_ok=True)
    final_dir.mkdir(parents=True, exist_ok=True)
    for f in to_download:
        dest = staging_dir / f["name"]
        download_file(
            f["url"],
            dest,
            expected_size=f["size"],
            desc=f"{cfg['local_dir']}/{f['name']}",
        )
        if flatten and staging_dir != final_dir:
            shutil.move(str(dest), str(final_dir / f["name"]))

    if flatten:
        if staging_dir != final_dir and staging_dir.exists():
            try:
                staging_dir.rmdir()
            except OSError:
                pass  # leftover files from a previous version, harmless
        _prune_stale_files(final_dir, remote_names)
        marker_path.write_text(version)

    return result


def sync_metabrainz(name, cfg, dry_run=False):
    targets = resolve_targets(cfg)
    details = [sync_target(cfg, t, dry_run=dry_run) for t in targets]
    return {
        "name": name,
        "status": "ok",
        "files": sum(d["files"] for d in details),
        "bytes": sum(d["bytes"] for d in details),
        "targets": details,
    }

In [ ]:
def _kaggle_obj_to_dict(obj):
    for method_name in ("to_dict", "model_dump", "dict"):
        method = getattr(obj, method_name, None)
        if callable(method):
            try:
                d = method()
                if isinstance(d, dict):
                    return d
            except Exception:
                pass
    d = getattr(obj, "__dict__", None)
    return d if isinstance(d, dict) else {}


def _kaggle_file_size(f):
    d = _kaggle_obj_to_dict(f)
    for key in (
        "totalBytes",
        "total_bytes",
        "sizeBytes",
        "size_bytes",
        "size",
        "fileSize",
        "file_size",
    ):
        val = d.get(key)
        if isinstance(val, (int, float)) and val:
            return int(val)
    return None


def sync_kaggle(name, cfg, dry_run=False):
    # KAGGLE_API_TOKEN, ~/.kaggle/access_token
    try:
        import kaggle

        api = kaggle.api
    except Exception as e:
        return {
            "name": name,
            "status": "needs_auth",
            "files": None,
            "bytes": None,
            "error": f"{type(e).__name__}: {e}",
        }

    dataset = cfg["kaggle_dataset"]
    try:
        files = api.dataset_list_files(dataset).files
    except Exception as e:
        return {
            "name": name,
            "status": "error",
            "files": None,
            "bytes": None,
            "error": f"{type(e).__name__}: {e}",
        }

    per_file_sizes = [_kaggle_file_size(f) for f in files]
    known = [s for s in per_file_sizes if s]
    total_bytes = sum(known) if known else None

    result = {"name": name, "status": "ok", "files": len(files), "bytes": total_bytes}
    if total_bytes is None and files:
        sample_fields = sorted(_kaggle_obj_to_dict(files[0]).keys())
        result["note"] = (
            "couldn't determine file sizes on this kaggle client version "
            f"(the download itself is unaffected). Fields on a file object: {sample_fields}"
        )

    if dry_run:
        return result

    local_dir = BASE_DIR / cfg["local_dir"]
    local_dir.mkdir(parents=True, exist_ok=True)

    # Kaggle's download API doesn't expose per-file resume the way the MetaBrainz side
    # does, so "already downloaded" is tracked with a completion marker written only after
    # a full, successful dataset_download_files() call, not just "the folder isn't
    # empty", which would also match a run that got interrupted halfway through and
    # quietly leave it that way forever (dataset_download_files re-downloads the whole zip
    # on retry, it can't pick up mid-file).
    marker_path = local_dir / ".complete"
    if marker_path.exists():
        result["files"] = 0
        result["bytes"] = 0
        result["note"] = "already downloaded, skipped"
        return result

    try:
        api.dataset_download_files(
            dataset, path=str(local_dir), unzip=True, quiet=False
        )
    except Exception as e:
        result["status"] = "error"
        result["error"] = f"{type(e).__name__}: {e}"
        return result

    marker_path.write_text("ok")
    return result


def sync_huggingface(name, cfg, dry_run=False):
    # Public dataset, no token required. snapshot_download already checks for
    # existing/complete files and skips or resumes them on its own.
    repo_id = cfg["hf_repo_id"]
    info = HfApi().dataset_info(repo_id, files_metadata=True)
    files = [s for s in info.siblings if s.size]
    total_bytes = sum(s.size for s in files)
    result = {"name": name, "status": "ok", "files": len(files), "bytes": total_bytes}

    if dry_run:
        return result

    local_dir = BASE_DIR / cfg["local_dir"]
    local_dir.mkdir(parents=True, exist_ok=True)
    pre_existing = any(local_dir.iterdir())

    result["path"] = snapshot_download(
        repo_id=repo_id, repo_type="dataset", local_dir=str(local_dir)
    )

    # snapshot_download() only transfers what's missing or changed, but files/bytes above
    # are the full dataset size either way. On a rerun where most of it was already
    # there, that would otherwise overstate what this run actually moved.
    if pre_existing:
        result["note"] = (
            "some files may already have been present, so the totals above are the full dataset size, not just what this run downloaded"
        )
    return result


def sync_source(name, cfg, dry_run=False):
    if not cfg.get("enabled", True):
        return {"name": name, "status": "disabled", "files": 0, "bytes": 0}
    kind = cfg["kind"]
    try:
        if kind == "metabrainz":
            return sync_metabrainz(name, cfg, dry_run=dry_run)
        if kind == "kaggle":
            return sync_kaggle(name, cfg, dry_run=dry_run)
        if kind == "huggingface":
            return sync_huggingface(name, cfg, dry_run=dry_run)
        raise ValueError(f"Unknown kind: {kind}")
    except Exception as e:
        return {
            "name": name,
            "status": "error",
            "files": None,
            "bytes": None,
            "error": f"{type(e).__name__}: {e}",
        }

## Dry run

Lists what would be downloaded and the total size for every source. Run this first, review the total, and disable anything not needed in the `SOURCES` config above. Kaggle shows "auth required" until credentials are added (see the `sync_kaggle` cell above).

In [ ]:
print(f"{'Source':30s} {'Kind':13s} {'Files':>10s} {'Approx. size':>15s}")
print("-" * 73)

dry_run_results = {}
grand_total = 0
for name, cfg in SOURCES.items():
    res = sync_source(name, cfg, dry_run=True)
    dry_run_results[name] = res
    kind = cfg.get("kind", "?")
    if res["status"] == "disabled":
        print(f"{name:30s} {kind:13s} {'--':>10s} {'disabled':>15s}")
        continue
    if res["status"] == "needs_auth":
        print(f"{name:30s} {kind:13s} {'--':>10s} {'auth required':>15s}")
        if res.get("error"):
            print(f"    -> {res['error']}")
        continue
    if res["status"] == "error":
        print(f"{name:30s} {kind:13s} {'--':>10s} {'error':>15s}")
        if res.get("error"):
            print(f"    -> {res['error']}")
        continue
    grand_total += res["bytes"] or 0
    files_str = str(res["files"]) if res["files"] is not None else "?"
    print(f"{name:30s} {kind:13s} {files_str:>10s} {human_size(res['bytes']):>15s}")
    if res.get("note"):
        print(f"    -> {res['note']}")

print("-" * 73)
print(f"Estimated total: {human_size(grand_total)}")

## Download

Only runs once `CONFIRM_DOWNLOAD` is set to `True`. Safe to re-run at any point: files
that already exist locally (or match the recorded "latest" version) are skipped, and any
partially-downloaded file resumes automatically.

In [ ]:
CONFIRM_DOWNLOAD = False  # set to True after a review of the dry-run totals above

if not CONFIRM_DOWNLOAD:
    print(
        "Nothing downloaded. Set CONFIRM_DOWNLOAD = True above and re-run this cell when ready."
    )
else:
    overall_files = 0
    overall_bytes = 0
    for name, cfg in SOURCES.items():
        print(f"\n{name} ({cfg.get('kind')})")
        result = sync_source(name, cfg, dry_run=False)

        if result["status"] == "disabled":
            print("  disabled, skipped")
            continue
        if result["status"] == "needs_auth":
            print(f"  skipped: Kaggle auth failed: {result.get('error')}")
            continue
        if result["status"] == "error":
            print(f"  error: {result.get('error')}")
            if result.get("files") is not None:
                print(
                    f"  (found {result['files']} file(s), {human_size(result.get('bytes'))} before the failure)"
                )
            continue

        if cfg.get("kind") == "metabrainz":
            for t in result["targets"]:
                tag = t["version"] or "(root)"
                if t["up_to_date"]:
                    print(f"  [{tag}] already up to date")
                else:
                    print(f"  [{tag}] {t['files']} file(s), {human_size(t['bytes'])}")
        else:
            note = f" ({result['note']})" if result.get("note") else ""
            print(f"  {result['files']} file(s), {human_size(result['bytes'])}{note}")

        overall_files += result["files"] or 0
        overall_bytes += result["bytes"] or 0

    print(
        f"\nDone. Downloaded {overall_files} file(s), {human_size(overall_bytes)} total."
    )

## Notes

### Resuming:
Re-run the notebook. Completed files are detected and skipped, `.part` files resume from where they stopped.

### Disk layout in the data folder:
- `musicbrainz/fullexport/`
- `musicbrainz/json-dumps/`
- `listenbrainz/fullexport/`
- `critiquebrainz/dump/`
- `canonical_data/`

^ Are the flattened folder structure. they always contain the latest version's files directly (files left over from a superseded version are pruned once the new one is fully in), plus a `.version` file recording which dump that is. 

- `listenbrainz/incremental/`

^ keeps every dated dump in its own subfolder. 

- `critiquebrainz/json/`
- `mlhd/`

^ Have no versioning on the server, so files sit directly in their folder.

- `kaggle/spotify-catalogue/`

^ has its own hidden `.complete` marker, written only after a full successful download (Kaggle's API can't resume a dataset file-by-file the way the MetaBrainz sources can, so this is what stands in for that).

- `huggingface/spotify-audio-features/`

^ relies on `snapshot_download`'s own resume/skip logic.

### Kaggle credentials:

Needed only for `kaggle_spotify_catalogue`. Generate a token at https://www.kaggle.com/settings/api -> "Generate New Token" or something along those lines, then run the 3rd option they have, where it saves your credentials locally with a chmod 600.

### Checksums:

MetaBrainz dumps include `MD5SUMS` / `SHA256SUMS` files. Use the helper below to verify a downloaded folder against its `SHA256SUMS` (this reads every file fully, so it can be slow for the multi-GB dumps).

In [ ]:
def verify_sha256sums(dir_path, sums_filename="SHA256SUMS"):
    dir_path = Path(dir_path)
    sums_file = dir_path / sums_filename
    if not sums_file.exists():
        print(f"No {sums_filename} in {dir_path}")
        return
    ok, bad, missing = [], [], []
    for line in sums_file.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        expected_hash, fname = line.split(maxsplit=1)
        fname = fname.lstrip("*")
        fpath = dir_path / fname
        if not fpath.exists():
            missing.append(fname)
            continue
        h = hashlib.sha256()
        with open(fpath, "rb") as fh:
            for chunk in iter(lambda: fh.read(CHUNK_SIZE), b""):
                h.update(chunk)
        (ok if h.hexdigest() == expected_hash else bad).append(fname)
    print(f"OK: {len(ok)}, mismatched: {len(bad)}, missing: {len(missing)}")
    if bad:
        print("Mismatched:", bad)
    if missing:
        print("Missing:", missing)


# Example:
# verify_sha256sums(BASE_DIR / "musicbrainz" / "fullexport")